# Notebook 01 — Setup & Data Exploration
## VisDrone MOT: Classical ML vs Deep Learning ReID Pipeline

### Project Overview
This project implements and compares **two Re-Identification (ReID) pipelines** for Multi-Object Tracking (MOT) on drone footage:

| Pipeline | Features | Matcher | Tracker |
|---|---|---|---|
| **Classical ML** | HSV histogram + LBP | XGBoost | SORT + appearance |
| **Deep Learning** | ResNet50 (2048-d) | Cosine similarity | DeepSORT-style |

### Key Research Question
> *When frames are deliberately dropped (simulating detection failure), which pipeline better preserves person identity across the gap?*

### VisDrone Annotation Format
Each row in a `.txt` annotation file:
```
frame_id, target_id, x, y, w, h, score, class, truncation, occlusion
```
- `frame_id`: 1-indexed frame number
- `target_id`: unique object ID within sequence (0 = ignore)
- `x, y, w, h`: bounding box in pixels (top-left corner + width + height)
- `score`: 0 = ignored region, 1 = valid detection
- `class`: 0=ignored, 1=pedestrian, 2=people, 3=bicycle, 4=car, 5=van, 6=truck, 7=tricycle, 8=awning-tricycle, 9=bus, 10=motor, 11=others
- `truncation`: 0=no truncation, 1=truncated
- `occlusion`: 0=none, 1=partial, 2=heavy

## Cell 1 — Install Required Libraries

In [ ]:
# Install all libraries needed for both pipelines
# Run this cell once — takes 2-3 minutes
import subprocess

packages = [
    'xgboost',          # Classical ML ReID matcher
    'scikit-image',     # LBP (Local Binary Pattern) feature extraction
    'scikit-learn',     # StandardScaler, metrics
    'scipy',            # Hungarian algorithm (linear_sum_assignment)
    'pandas',           # DataFrames for annotations
    'matplotlib',       # Plotting
    'seaborn',          # Enhanced plots
    'tqdm',             # Progress bars
    'Pillow',           # Image loading
    'opencv-python',    # Image processing, HSV conversion
    'lap',              # Fast Hungarian algorithm (optional, falls back to scipy)
]

for pkg in packages:
    result = subprocess.run(
        ['pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '⚠️'
    print(f'{status} {pkg}')

print('\nAll packages done!')

## Cell 2 — Import Libraries & Check GPU

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from pathlib import Path
from collections import defaultdict

print('='*55)
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import cv2, sklearn, xgboost
print(f'OpenCV       : {cv2.__version__}')
print(f'scikit-learn : {sklearn.__version__}')
print(f'XGBoost      : {xgboost.__version__}')
print('='*55)

## Cell 3 — Define Dataset Paths

In [ ]:
# ── UPDATE THESE PATHS TO MATCH YOUR MACHINE ──────────────────────
BASE_DIR   = r'D:\MTech\Sem-2\IT585-Advanced_ML\Project\AML-project'
DATA_DIR   = os.path.join(BASE_DIR, 'data', 'VisDrone_dataset')
TRAIN_DIR  = os.path.join(DATA_DIR, 'VisDrone2019-MOT-train')
VAL_DIR    = os.path.join(DATA_DIR, 'VisDrone2019-MOT-val')
OUTPUT_DIR = os.path.join(BASE_DIR, 'visdrone_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'splits'),    exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'features'),  exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'models'),    exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tracks'),    exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'reports'),   exist_ok=True)
# ──────────────────────────────────────────────────────────────────

# VisDrone class labels (class id → name)
CLASS_NAMES = {
    0: 'ignored', 1: 'pedestrian', 2: 'people',
    3: 'bicycle', 4: 'car', 5: 'van',
    6: 'truck', 7: 'tricycle', 8: 'awning-tricycle',
    9: 'bus', 10: 'motor', 11: 'others'
}

# Classes we care about for tracking (pedestrians + vehicles)
VALID_CLASSES = {1, 2, 4, 5, 6, 9}  # pedestrian, people, car, van, truck, bus

for name, path in [('TRAIN', TRAIN_DIR), ('VAL', VAL_DIR), ('OUTPUT', OUTPUT_DIR)]:
    status = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'{name:8s}: {status}')
    print(f'         {path}')

## Cell 4 — Core Annotation Reader

### VisDrone Annotation Format
```
frame_id, target_id, x, y, w, h, score, class_id, truncation, occlusion
```
### Filtering Rules
We keep a detection if ALL of the following hold:
- `score = 1` (not an ignored region)
- `target_id > 0` (has a valid identity label)
- `class_id ∈ VALID_CLASSES` (is a class we care about)
- `w > 0` and `h > 0` (has a non-degenerate bounding box)

In [ ]:
def read_visdrone_annotation(anno_path, valid_classes=None):
    """
    Read a single VisDrone annotation .txt file.

    Format per row:
        frame_id, target_id, x, y, w, h, score, class_id, truncation, occlusion

    Args:
        anno_path    : path to the .txt annotation file
        valid_classes: set of class IDs to keep (None = keep all)

    Returns:
        pd.DataFrame with columns:
            frame_id, target_id, x, y, w, h, score, class_id, truncation, occlusion
        — filtered to only valid, labelled detections
    """
    cols = ['frame_id', 'target_id', 'x', 'y', 'w', 'h',
            'score', 'class_id', 'truncation', 'occlusion']

    df = pd.read_csv(anno_path, header=None, names=cols)

    # Filter Rule 1: score must be 1 (ignore score=0 regions)
    df = df[df['score'] == 1]

    # Filter Rule 2: target_id must be > 0 (0 = background/ignored)
    df = df[df['target_id'] > 0]

    # Filter Rule 3: valid object class
    if valid_classes is not None:
        df = df[df['class_id'].isin(valid_classes)]

    # Filter Rule 4: non-degenerate bounding box
    df = df[(df['w'] > 0) & (df['h'] > 0)]

    return df.reset_index(drop=True)


def get_sequence_names(split_dir):
    """Return sorted list of sequence names from a VisDrone split directory."""
    anno_dir = os.path.join(split_dir, 'annotations')
    return sorted([f.replace('.txt', '') for f in os.listdir(anno_dir)
                   if f.endswith('.txt')])


# Test on first training sequence
train_seqs = get_sequence_names(TRAIN_DIR)
val_seqs   = get_sequence_names(VAL_DIR)

print(f'Training sequences : {len(train_seqs)}')
print(f'Validation sequences: {len(val_seqs)}')
print(f'\nFirst 5 training sequences:')
for s in train_seqs[:5]: print(f'  {s}')

# Quick test
sample_seq = train_seqs[0]
sample_anno_path = os.path.join(TRAIN_DIR, 'annotations', sample_seq + '.txt')
sample_df = read_visdrone_annotation(sample_anno_path, VALID_CLASSES)
print(f'\nSample sequence: {sample_seq}')
print(f'Total valid detections: {len(sample_df)}')
print(f'Unique frames: {sample_df["frame_id"].nunique()}')
print(f'Unique target IDs: {sample_df["target_id"].nunique()}')
print(f'\nSample rows:')
print(sample_df.head())

## Cell 5 — Dataset Statistics

In [ ]:
from tqdm import tqdm

def compute_dataset_stats(split_dir, seqs, valid_classes):
    """Compute summary statistics across all sequences in a split."""
    stats = []
    for seq in tqdm(seqs, desc='Computing stats'):
        anno_path = os.path.join(split_dir, 'annotations', seq + '.txt')
        seq_dir   = os.path.join(split_dir, 'sequences', seq)
        df = read_visdrone_annotation(anno_path, valid_classes)

        # Count images in sequence folder
        n_frames = len([f for f in os.listdir(seq_dir)
                        if f.endswith('.jpg')])

        stats.append({
            'sequence'    : seq,
            'n_frames'    : n_frames,
            'n_detections': len(df),
            'n_ids'       : df['target_id'].nunique(),
            'classes'     : sorted(df['class_id'].unique().tolist()),
            'avg_det_per_frame': len(df) / max(1, n_frames)
        })
    return pd.DataFrame(stats)

print('Computing training split statistics...')
train_stats = compute_dataset_stats(TRAIN_DIR, train_seqs, VALID_CLASSES)
print(train_stats[['sequence','n_frames','n_detections','n_ids','avg_det_per_frame']]
      .to_string(index=False))

print(f'\n=== TRAINING SET SUMMARY ===')
print(f'Sequences    : {len(train_stats)}')
print(f'Total frames : {train_stats["n_frames"].sum():,}')
print(f'Total dets   : {train_stats["n_detections"].sum():,}')
print(f'Total IDs    : {train_stats["n_ids"].sum():,}')
print(f'Avg frames/seq: {train_stats["n_frames"].mean():.0f}')

## Cell 6 — Visualise a Sequence with Bounding Boxes

In [ ]:
def visualise_sequence_frames(split_dir, seq_name, frame_ids=None,
                               valid_classes=None, n_frames=4):
    """
    Display sample frames from a sequence with ground truth bounding boxes.
    Each identity gets a consistent colour across frames.

    Args:
        split_dir    : path to train or val directory
        seq_name     : sequence folder name
        frame_ids    : specific frame IDs to show (None = auto-select)
        valid_classes: filter to these class IDs
        n_frames     : number of frames to display
    """
    anno_path = os.path.join(split_dir, 'annotations', seq_name + '.txt')
    seq_dir   = os.path.join(split_dir, 'sequences', seq_name)

    df = read_visdrone_annotation(anno_path, valid_classes)
    all_frame_ids = sorted(df['frame_id'].unique())

    # Auto-select evenly spaced frames if not specified
    if frame_ids is None:
        indices   = np.linspace(0, len(all_frame_ids)-1, n_frames, dtype=int)
        frame_ids = [all_frame_ids[i] for i in indices]

    # Assign consistent colour per target_id
    unique_ids = df['target_id'].unique()
    rng = np.random.RandomState(42)
    id_colours = {tid: tuple(rng.randint(50, 230, 3).tolist())
                  for tid in unique_ids}

    fig, axes = plt.subplots(1, len(frame_ids), figsize=(5*len(frame_ids), 4))
    if len(frame_ids) == 1: axes = [axes]
    fig.suptitle(f'Sequence: {seq_name}', fontsize=13, fontweight='bold')

    for ax, fid in zip(axes, frame_ids):
        # Load image — VisDrone images are named 0000001.jpg (7 digits)
        img_path = os.path.join(seq_dir, f'{fid:07d}.jpg')
        if not os.path.exists(img_path):
            ax.set_title(f'Frame {fid} not found')
            continue

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)

        # Draw bounding boxes for this frame
        frame_df = df[df['frame_id'] == fid]
        for _, row in frame_df.iterrows():
            col = [c/255 for c in id_colours[row['target_id']]]
            rect = patches.Rectangle(
                (row['x'], row['y']), row['w'], row['h'],
                linewidth=1.5, edgecolor=col, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(row['x'], row['y']-3, f"ID:{row['target_id']}",
                    fontsize=6, color=col, fontweight='bold')

        ax.set_title(f'Frame {fid} | {len(frame_df)} dets', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Visualise first training sequence
print(f'Visualising: {train_seqs[0]}')
visualise_sequence_frames(TRAIN_DIR, train_seqs[0], valid_classes=VALID_CLASSES)

## Cell 7 — Save Global Config
Save all paths and configuration to a shared config file used by all notebooks.

In [ ]:
import json

config = {
    'BASE_DIR'      : BASE_DIR,
    'DATA_DIR'      : DATA_DIR,
    'TRAIN_DIR'     : TRAIN_DIR,
    'VAL_DIR'       : VAL_DIR,
    'OUTPUT_DIR'    : OUTPUT_DIR,
    'VALID_CLASSES' : list(VALID_CLASSES),
    'CLASS_NAMES'   : CLASS_NAMES,
    'TRAIN_SEQS'    : train_seqs,
    'VAL_SEQS'      : val_seqs,
    # Hyperparameters
    'HSV_BINS'      : [16, 16, 8],   # H=16, S=16, V=8 bins
    'LBP_POINTS'    : 8,
    'LBP_RADIUS'    : 1,
    'EMA_ALPHA'     : 0.9,
    'MAX_AGE'       : 20,
    'MIN_HITS'      : 3,
    'IOU_THRESHOLD' : 0.3,
    'W_IOU'         : 0.5,
    'W_APP'         : 0.5,
}

config_path = os.path.join(OUTPUT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✅ Config saved to: {config_path}')
print('\nConfig contents:')
for k, v in config.items():
    if isinstance(v, list) and len(v) > 5:
        print(f'  {k:20s}: [{v[0]}, ..., {v[-1]}] ({len(v)} items)')
    else:
        print(f'  {k:20s}: {v}')
print('\n✅ Ready for Notebook 02: Data Preparation')